# EduTech Global · 06 · Workforce Stability Dashboard (Deliverable 2)

**Deliverable 2.** Excel dashboard summarising attrition findings + recommended HR interventions with cost/impact estimates. Designed to be opened in front of an HR/CFO audience.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:


from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "edutech":
            return parent
        if (parent / "scripts").exists() and (parent / "outputs").exists() and (parent / "data").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Allow either dataset/ or data/ to hold the source CSV; accept both filenames
HR_CSV_NAMES = ["WA_Fn-UseC_-HR-Employee-Attrition.csv", "HR-Employee-Attrition.csv"]
hr_locations = []
for name in HR_CSV_NAMES:
    hr_locations.extend([DATASET_DIR / name, DATA_DIR / name])
HR_CSV_PATH = next((p for p in hr_locations if p.exists()), None)
assert HR_CSV_PATH is not None, (
    f"Could not find HR CSV. Looked for {HR_CSV_NAMES} in {DATASET_DIR} and {DATA_DIR}"
)
print(f"HR data file : {HR_CSV_PATH}")


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter

# Load the analysed HR data
hr = pd.read_csv(DATA_DIR / "hr_with_features.csv")
hr["AttritionFlag"] = (hr["Attrition"] == "Yes").astype(int)
overall = hr["AttritionFlag"].mean()

# Pre-compute key roll-ups (so dashboard cells can reference these as hard values where needed)
total_emp = len(hr)
leavers = int(hr["AttritionFlag"].sum())
ot_rate_yes = hr[hr["OverTime"]=="Yes"]["AttritionFlag"].mean()
ot_rate_no  = hr[hr["OverTime"]=="No"]["AttritionFlag"].mean()
junior_attr = hr[hr["JobLevel"]<=2]["AttritionFlag"].mean()
new_hire_attr = hr[hr["YearsAtCompany"]<=1]["AttritionFlag"].mean()

print(f"Total employees   : {total_emp:,}")
print(f"Total leavers     : {leavers:,}")
print(f"Overall attrition : {overall:.2%}")
print(f"OverTime=Yes      : {ot_rate_yes:.2%}")
print(f"OverTime=No       : {ot_rate_no:.2%}")
print(f"Junior (lvl ≤ 2)  : {junior_attr:.2%}")
print(f"New hires (≤1 yr) : {new_hire_attr:.2%}")


In [ ]:
wb = Workbook()
ARIAL = lambda **kw: Font(name="Arial", **kw)
header_fill = PatternFill("solid", fgColor="1F3864")
kpi_fill    = PatternFill("solid", fgColor="E2EFDA")
red_fill    = PatternFill("solid", fgColor="FCE4D6")
amber_fill  = PatternFill("solid", fgColor="FFF2CC")
band_fill   = PatternFill("solid", fgColor="F2F2F2")
thin = Side(border_style="thin", color="BFBFBF")
box  = Border(left=thin, right=thin, top=thin, bottom=thin)

# === Sheet 1: Dashboard ===
dash = wb.active
dash.title = "Dashboard"

dash["A1"] = "EDUTECH GLOBAL · WORKFORCE STABILITY DASHBOARD"
dash["A1"].font = ARIAL(size=18, bold=True, color="FFFFFF")
dash["A1"].fill = header_fill
dash["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
dash.row_dimensions[1].height = 30
dash.merge_cells("A1:H1")

# KPI row
dash["A3"] = "HEADLINE KPIs"
dash["A3"].font = ARIAL(size=12, bold=True, color="1F3864")

kpis = [
    ("Total employees",         f"{total_emp:,}",          "FFFFFF"),
    ("Total leavers (period)",  f"{leavers:,}",            "FFFFFF"),
    ("Attrition rate",          f"{overall:.1%}",          "FCE4D6"),
    ("Industry benchmark",      "10-15%",                  "FFFFFF"),
    ("Variance vs benchmark",   f"+{(overall-0.125)*100:.1f}pp", "FCE4D6"),
]
for j, (lbl, val, fill_hex) in enumerate(kpis):
    col = j * 2 + 1   # spread across cols 1,3,5,7,9
    dash.cell(row=4, column=col, value=lbl).font = ARIAL(size=10, bold=True, color="595959")
    cell = dash.cell(row=5, column=col, value=val)
    cell.font = ARIAL(size=18, bold=True, color="1F3864")
    cell.fill = PatternFill("solid", fgColor=fill_hex)
    cell.border = box
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Section 2 — Top Drivers
dash["A8"] = "TOP 5 ATTRITION DRIVERS"
dash["A8"].font = ARIAL(size=12, bold=True, color="1F3864")

drivers = pd.read_csv(OUTPUTS_DIR / "attrition_drivers.csv")
top5 = drivers.head(5).reset_index(drop=True)

drv_headers = ["Rank", "Driver", "Test", "Strength", "p-value", "Significant"]
for j, h in enumerate(drv_headers, start=1):
    c = dash.cell(row=9, column=j, value=h)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
    c.alignment = Alignment(horizontal="center")

for i, row in top5.iterrows():
    r = 10 + i
    dash.cell(row=r, column=1, value=i+1).alignment = Alignment(horizontal="center")
    dash.cell(row=r, column=2, value=row["feature"])
    dash.cell(row=r, column=3, value=row["test"])
    dash.cell(row=r, column=4, value=row["statistic"]).number_format = "0.000"
    dash.cell(row=r, column=5, value=row["p_value"]).number_format = "0.0000"
    dash.cell(row=r, column=6, value="Yes" if row["significant"] else "No")
    for c in range(1, 7):
        dash.cell(row=r, column=c).border = box
        dash.cell(row=r, column=c).font = ARIAL(size=10)
        if row["significant"]:
            dash.cell(row=r, column=c).fill = red_fill

# Section 3 — Risk Segments
dash["A17"] = "HIGH-RISK SEGMENTS"
dash["A17"].font = ARIAL(size=12, bold=True, color="1F3864")

seg_headers = ["Segment", "Headcount", "Attrition rate", "Lift vs avg"]
for j, h in enumerate(seg_headers, start=1):
    c = dash.cell(row=18, column=j, value=h)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box

segments = [
    ("OverTime = Yes",                      hr[hr["OverTime"]=="Yes"].shape[0],
                                            ot_rate_yes,      ot_rate_yes/overall),
    ("Junior (Job Level ≤ 2)",              hr[hr["JobLevel"]<=2].shape[0],
                                            junior_attr,      junior_attr/overall),
    ("New hires (≤ 1 year)",                hr[hr["YearsAtCompany"]<=1].shape[0],
                                            new_hire_attr,    new_hire_attr/overall),
    ("Single, junior, OverTime",
        hr[(hr["MaritalStatus"]=="Single") & (hr["JobLevel"]<=2) & (hr["OverTime"]=="Yes")].shape[0],
        hr[(hr["MaritalStatus"]=="Single") & (hr["JobLevel"]<=2) & (hr["OverTime"]=="Yes")]["AttritionFlag"].mean(),
        (hr[(hr["MaritalStatus"]=="Single") & (hr["JobLevel"]<=2) & (hr["OverTime"]=="Yes")]["AttritionFlag"].mean())/overall),
]

for i, (name, hc, rate, lift) in enumerate(segments):
    r = 19 + i
    dash.cell(row=r, column=1, value=name)
    dash.cell(row=r, column=2, value=hc).number_format = "#,##0"
    dash.cell(row=r, column=3, value=rate).number_format = "0.0%"
    dash.cell(row=r, column=4, value=lift).number_format = '0.0"x"'
    for c in range(1, 5):
        dash.cell(row=r, column=c).border = box
        dash.cell(row=r, column=c).font = ARIAL(size=10)
        if lift > 1.5:
            dash.cell(row=r, column=c).fill = red_fill
        elif lift > 1.2:
            dash.cell(row=r, column=c).fill = amber_fill

# Section 4 — Recommended interventions
dash["A25"] = "RECOMMENDED HR INTERVENTIONS"
dash["A25"].font = ARIAL(size=12, bold=True, color="1F3864")

int_headers = ["#", "Intervention", "Targets driver", "Annual cost (USD)",
               "Expected attrition reduction (pp)", "Annual replacement-cost saving (USD)"]
for j, h in enumerate(int_headers, start=1):
    c = dash.cell(row=26, column=j, value=h)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
    c.alignment = Alignment(horizontal="center", wrap_text=True)
dash.row_dimensions[26].height = 36

# Replacement cost = ~6 months salary per leaver (industry benchmark)
# 6 * mean(MonthlyIncome) * (employees * pp_reduction)
mean_monthly = hr["MonthlyIncome"].mean()
replace_cost_per_leaver = 6 * mean_monthly

interventions = [
    ("OverTime cap policy (max 8 hrs/wk overtime)",
     "OverTime", 100_000, 3.0),
    ("Mentor pairing for new hires (first 12 months)",
     "Tenure ≤ 1 yr", 80_000, 2.5),
    ("Career-path clarity workshop for Job Levels 1-2",
     "JobLevel ≤ 2", 60_000, 1.8),
    ("Quarterly stay-interview programme",
     "All segments", 50_000, 1.0),
    ("Hybrid/commute-flex policy (distance > 20km)",
     "DistanceFromHome", 40_000, 0.8),
]
for i, (name, target, cost, pp_reduction) in enumerate(interventions):
    r = 27 + i
    saving = (pp_reduction / 100) * total_emp * replace_cost_per_leaver
    dash.cell(row=r, column=1, value=i+1).alignment = Alignment(horizontal="center")
    dash.cell(row=r, column=2, value=name)
    dash.cell(row=r, column=3, value=target)
    dash.cell(row=r, column=4, value=cost).number_format = "$#,##0"
    dash.cell(row=r, column=5, value=pp_reduction).number_format = '0.0" pp"'
    dash.cell(row=r, column=6, value=saving).number_format = "$#,##0"
    for c in range(1, 7):
        dash.cell(row=r, column=c).border = box
        dash.cell(row=r, column=c).font = ARIAL(size=10)
    dash.cell(row=r, column=6).font = ARIAL(size=10, bold=True, color="00703C")

# Total row
r = 32
dash.cell(row=r, column=1, value="").alignment = Alignment(horizontal="center")
dash.cell(row=r, column=2, value="TOTAL programme").font = ARIAL(size=11, bold=True)
dash.cell(row=r, column=4, value=f"=SUM(D27:D31)").number_format = "$#,##0"
dash.cell(row=r, column=5, value=f"=SUM(E27:E31)").number_format = '0.0" pp"'
dash.cell(row=r, column=6, value=f"=SUM(F27:F31)").number_format = "$#,##0"
for c in range(1, 7):
    dash.cell(row=r, column=c).border = box
    dash.cell(row=r, column=c).fill = kpi_fill
    dash.cell(row=r, column=c).font = ARIAL(size=11, bold=True)
dash.cell(row=r, column=6).font = ARIAL(size=12, bold=True, color="00703C")

# Column widths
widths = [4, 36, 24, 14, 22, 26]
for j, w in enumerate(widths, start=1):
    dash.column_dimensions[get_column_letter(j)].width = w

print("Dashboard sheet built")


In [ ]:
# === Sheet 2: Detail tables ===
det = wb.create_sheet("Detail Tables")
det["A1"] = "DETAILED ATTRITION TABLES"
det["A1"].font = ARIAL(size=14, bold=True, color="1F3864")
det.merge_cells("A1:F1")

# Pull tenure summary
tenure = pd.read_csv(OUTPUTS_DIR / "attrition_by_tenure.csv")
det["A3"] = "Attrition by Tenure"
det["A3"].font = ARIAL(size=12, bold=True, color="1F3864")
for j, col in enumerate(tenure.columns, start=1):
    c = det.cell(row=4, column=j, value=col)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
for i, row in tenure.iterrows():
    for j, col in enumerate(tenure.columns, start=1):
        c = det.cell(row=5+i, column=j, value=row[col])
        c.border = box; c.font = ARIAL(size=10)
        if "rate" in col.lower():
            c.number_format = "0.0%" if col == "attrition_rate" else '0.0"%"'

# Spacer + income
income = pd.read_csv(OUTPUTS_DIR / "attrition_by_income.csv")
start_row = 5 + len(tenure) + 3
det.cell(row=start_row-1, column=1, value="Attrition by Income Quintile").font = ARIAL(size=12, bold=True, color="1F3864")
for j, col in enumerate(income.columns, start=1):
    c = det.cell(row=start_row, column=j, value=col)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
for i, row in income.iterrows():
    for j, col in enumerate(income.columns, start=1):
        c = det.cell(row=start_row+1+i, column=j, value=row[col])
        c.border = box; c.font = ARIAL(size=10)

# Spacer + job role
role = pd.read_csv(OUTPUTS_DIR / "attrition_by_jobrole.csv")
start_row += len(income) + 3
det.cell(row=start_row-1, column=1, value="Attrition by Job Role").font = ARIAL(size=12, bold=True, color="1F3864")
for j, col in enumerate(role.columns, start=1):
    c = det.cell(row=start_row, column=j, value=col)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
for i, row in role.iterrows():
    for j, col in enumerate(role.columns, start=1):
        c = det.cell(row=start_row+1+i, column=j, value=row[col])
        c.border = box; c.font = ARIAL(size=10)

for col in "ABCDEF":
    det.column_dimensions[col].width = 22

# SAVE
out_file = OUTPUTS_DIR / "Workforce_Stability_Dashboard.xlsx"
wb.save(out_file)
print(f"Saved: {out_file}")


✅ **Notebook 06 complete.** Move to `07_build_market_entry_playbook.ipynb` (final report).